# Scorer: embeddings, similarity baselines, trained heads

The main model of `../docs/scorer.md` on the final dataset (`cryandrrich/editjudge`):
1. **Embed** every image and text once with 5 frozen encoders (CLIP ViT-L/14, PickScore, SigLIP, HPSv2.1,
   DINOv2). Every image is decoded once for all encoders, padded to a square (no centre crop), and both GPUs
   work at once, each on half of the images (`python -m src.scorer embed --gpus 2`).
2. **Similarity baselines** from the embeddings: CLIP-score, PickScore, SigLIP, HPSv2.1 (cosine of edit and
   instruction), and the human ceiling.
3. **Train** every run of `configs/scorer.toml`, 5 seeds each (seconds per head on a GPU).
4. **Report**: `out/scorer/report.md` (the run selected by dev, every baseline, paired differences).

Time: about 1 h, most of it the embeddings (and about 5 GB of model downloads).

How to run:
1. Input: the dataset `cryandrrich/editjudge`. Settings: GPU T4 x2, internet on. Push the code first.
2. Run it with **Save Version > Save & Run All (Commit)**.
3. To train again without embedding again (e.g. after changing `configs/scorer.toml`): **Add Input > Your
   Work > this notebook's latest version**. The setup cell copies its embeddings, and `embed` skips every
   encoder whose embeddings already cover the dataset.
4. The reward-model baselines come from `scorer_baselines.ipynb`. To put them in the same report, add that
   notebook's latest version as input too: its score files are copied before the report.
5. Download `out/scorer_results.zip` (small: metrics, per-edit scores, report).

In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_dataset():
    """The input folder that holds pairs.jsonl and images/ (the final dataset)."""
    for pattern in ("/kaggle/input/*/pairs.jsonl", "/kaggle/input/*/*/pairs.jsonl", "/kaggle/input/*/*/*/pairs.jsonl"):
        found = sorted(glob.glob(pattern))
        if found:
            return os.path.dirname(found[0])
    raise SystemExit("no input with pairs.jsonl: add the dataset cryandrrich/editjudge")

def find_previous(subfolder, pattern):
    """The earlier output folder out/scorer/<subfolder> (a version of this notebook added as input) with the most
    files matching `pattern`, or None."""
    best, best_count = None, 0
    # Kaggle mounts an input at a varying depth, e.g. /kaggle/input/notebooks/<owner>/<notebook>/out/...
    folders = [folder for depth in range(6) for folder in glob.glob("/kaggle/input/" + "*/" * depth + f"scorer/{subfolder}")]
    for folder in folders:
        count = len(glob.glob(f"{folder}/{pattern}", recursive=True))
        if count > best_count:
            best, best_count = folder, count
    return best

DATA = find_dataset()
pair_lines = [json.loads(line) for line in open(f"{DATA}/pairs.jsonl")]
print("dataset:", DATA, "-", len(pair_lines), "pairs,", sum(bool(line.get("instruction")) for line in pair_lines),
      "with an instruction")
if not any(line.get("instruction") for line in pair_lines):
    raise SystemExit("this dataset version has no instructions: add the latest version of cryandrrich/editjudge")
os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.environ["EJ_ENVS_DIR"] = "/tmp/ej-envs"  # one Python environment per group of conflicting packages
os.makedirs(f"{OUT}/scorer", exist_ok=True)
os.chdir(CODE)

cached = find_previous("embeddings", "*/images*.npy")
if cached:
    shutil.copytree(cached, f"{OUT}/scorer/embeddings", dirs_exist_ok=True)
    print("embeddings copied from", cached)
rewards = find_previous("baselines", "*.jsonl")
if rewards:
    os.makedirs(f"{OUT}/scorer/baselines", exist_ok=True)
    for path in glob.glob(f"{rewards}/*"):
        shutil.copy(path, f"{OUT}/scorer/baselines/")
    print("reward-model scores copied from", rewards)

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets): faster downloads
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    os.environ["HF_HUB_VERBOSITY"] = "error"

In [ ]:
!pip -q install -U "transformers>=4.56,<5" open_clip_torch sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run_scorer(*args):
    """python -m src.scorer <args>: prints every line as it comes, and returns the output."""
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.scorer", *map(str, args)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)
    return "".join(lines)

run_scorer("list")

In [ ]:
SMOKE = False           # True: 100 samples and one run, to check the setup (a few minutes plus the downloads)
GPUS = 2

In [ ]:
run_scorer("embed", "--gpus", GPUS, *(["--limit", 100] if SMOKE else []))
for path in sorted(glob.glob(f"{OUT}/scorer/embeddings/*/meta*.json")):
    print(path.split("/")[-2], json.load(open(path))["seconds_per_image_all_backbones"], "s per image (all encoders)")

In [ ]:
run_scorer("baselines")
run_scorer("train", *(["--run", "mlp-siglip"] if SMOKE else []))
run_scorer("report")
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/scorer/report.md").read()))

In [ ]:
# Small files to bring back; the embeddings stay in this notebook's output (input for the next version)
import zipfile
keep = [f"{OUT}/scorer/{name}" for name in ("report.md", "report.json", "baselines.json")]
keep += glob.glob(f"{OUT}/scorer/baselines/*") + glob.glob(f"{OUT}/scorer/runs/*/metrics.json")
keep += glob.glob(f"{OUT}/scorer/runs/*/scores.jsonl") + glob.glob(f"{OUT}/scorer/embeddings/*/meta*.json")
keep += glob.glob(f"{OUT}/scorer/logs/*")
with zipfile.ZipFile(f"{OUT}/scorer_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in keep:
        if os.path.isfile(path):
            archive.write(path, os.path.relpath(path, OUT))
print(f"{OUT}/scorer_results.zip:", len(keep), "files")